## Boosting Machine Learning Models

### Boosting:

- Boosting ensemble method use weak learners as base models that are simple and tend to sufferfrom high bias. (*the weak learners underfit the data*)

- Boosting is a sequential learning technique where each of the base models builds off of the previous model, aiming to fix errors in the previous stage and improve the proformance of the final ensemble model.

- Boosted ensembling decisions:
- 1. Sequential Fitting Model
- 2. Aggregation Method

- Boosting Algorithms:
- 1. Adaptive Boosting
- 2. Gradient Boosting

- Can be applied to any machine learning model

---

## Adaptive Boosting (AdaBoost):
![AdaBoost Image](images/adaboost.png)

- Sequential ensembling method that can be used for both classification and regression. 
- Commonly used with decision trees, but can used with any based machine learning model.
    - **Sequential Fitting Method**: accomplished by updating the weight attached to each of the training dataset observations proceeding from one base model to the next.

    - **Aggregation Method**: a weighted sum of those base models where the model weight is dependent on the error of that particular estimator.

> The training of an AdaBoost model is the process of determining the training dataset observation weights at each step as well as the final weight for each base model for aggregation.

### Adaptive Boosting

1. Fit an estimator, the 1st Base Model. 
    - Should be very simple and tend to overfit.

> Each of the base models will contribute a different amount to the final ensemble model. The influence that a particular base model contributes is going to be dependent on the number of errors it makes, or for regression, the magnitude of the errors it makes. 

2. Once the Result of the 1st Base Model is evaluated, we can Weight the Model and assign it a value, here indicated by alpha_1.

3. Reweight the data to prepare for the next stage of the sequential process.
    - By assigning those misclassified points a larger weight, we are asking the the 2nd Base Model to give them preferential treatment during the Model Fitting.

4. Once again we assign the base model a weight, alpha_2 proportional to the errors it makes and prepare for the next stage of the sequential learning by reweighting the training data.

5. Once we have reached the predefined number of estimators for our AdaBoost model, the base models are ready to aggregate. In this example we have chosen n_estimators = 3. The influence of each base model in the final ensemble model will be proportional to the alpha it was assigned during the training process.

---

### Adaptive Boosting Implementation

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Load dataset to a pandas DataFrame
path_to_data = 'https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data'
column_names = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'accep']
df = pd.read_csv(path_to_data, names=column_names)

target_column = 'accep'
raw_feature_columns = [col for col in column_names if col != target_column]

# Create dummy variables from the feature columns
X = pd.get_dummies(df[raw_feature_columns], drop_first=True)

# Convert target column to binary variable; 0 if 'unacc', 1 otherwise
df[target_column] = np.where(df[target_column] == 'unacc', 0, 1)
y = df[target_column]

# Split the full dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=123, test_size=0.3)


# 1. Create a decision stump base model using the Decision Tree Classifier and print its parameters
decision_stump = DecisionTreeClassifier(max_depth=1)
decision_stump.fit(X_train, y_train)
print(decision_stump.get_params())

# 2. Create an Adaptive Boost Classifier and print its parameters
ada_classifier = AdaBoostClassifier(base_estimator=decision_stump, n_estimators=5)
print(ada_classifier.get_params())

# 3. Fit the Adaptive Boost Classifier to the training data and get the list of predictions
ada_classifier.fit(X_train, y_train)
y_pred = ada_classifier.predict(X_test)

# 4. Calculate the accuracy, precision, recall, and f1-score on the testing data
accuracy = accuracy_score(y_test, y_pred).round(4)
precision = precision_score(y_test, y_pred).round(4)
recall = recall_score(y_test, y_pred).round(4)
f1 = f1_score(y_test, y_pred).round(4)

print(f'Test set accuracy:\t{accuracy}')
print(f'Test set precision:\t{precision}')
print(f'Test set recall:\t{recall}')
print(f'Test set f1-score:\t{f1}')

# 5. Remove the comments from the following code block to print the confusion matrix
test_conf_matrix = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=[1, 0]), 
    index=['actual yes', 'actual no'], 
    columns=['predicted yes', 'predicted no']
)
print(f'Confusion Matrix:\n{test_conf_matrix.to_string()}')
```

---

## Gradient Boosting:
![gradient boosting](images/gradient_boosting.png)

- Sequentialensembling method that can be used for both classification and regression.

- Can be used on any machine learning model, most commonly used with decision trees, known as *Gradient Boosted Trees*. 

    - **Sequential Fittinf Method**: accomplished by fitting a base model to the negative gadient of the erro in the previous stage. 

    - **Aggregation Method**: a weighted sum of those base models where the model weight is constant.

> The training of a Gradient Boosted model is the process of determining the base model error at each step and using those to determine how to best formulate the subsequent base model.

### Gradient Boosting

1. fit an estimator. (base estimators for boosting algorithms tend to be simplpe and high bias (overfitting))
    - Can and tent to include more decision branches
    - Gradient Boosted trees will have up to 32 leaf nodes.(depth of 5 levels)

2. Once the 1st Base Model is trained, the residual errors (`h_1`), of the model given the training data are determined. The residual error is the difference between the actual and predicted values for each of the training data instances.

![y1 residual error](images/h1_y1_residual_error.png)

> The errors will be greater for the training data instances where the model did not do as good of a job with its prediction and will be lower on training data instances where the model fit the data well.

3. Fit the 2nd Base Model.
    - fit the model on the errors of the previous stage
    - The 2nd Base Model is literally learning from the mistakes of the 1st Base Model through those residuals that were calculated.

4. The results of the 2nd Base Model are multiplied by a constant learning rate, `alpha`, and added to the results of the 1st Base Model to give the set of updated predictions, The results of the second base model, which was tasked with fitting the errors of the first base model are multiplied by a constant learning rate, `alpha` and added to the results of the first base model to give us a set of updated predictions, `y_2(predicted)`.

    - *residual errors of the 2nd stage are calculated using the updated predictions*

![y2 residual error](images/h1_y2_residual_error.png)

5. The subsequent stages repeat the same steps. At stage N, the base model is fit on the errors calculated at the previous stage `h_(N-1)`. The new model that is fit is multiplied by the constant learning rate `alpha` and added to the predictions of the previous stage.

> Once we have reached the predefined number of estimators for our Gradient Boosting model or the residual errors are not changing between iterations, the model will stop training and we end up with the resultant ensemble model.

### Gradient Boosting Implementation

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Load dataset to a pandas DataFrame
path_to_data = 'https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data'
column_names = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'accep']

df = pd.read_csv(path_to_data, names=column_names)
target_column = 'accep'
raw_feature_columns = [col for col in column_names if col != target_column]

# Create dummy variables from the feature columns
X = pd.get_dummies(df[raw_feature_columns], drop_first=True)

# Convert target column to binary variable; 0 if 'unacc', 1 otherwise
df[target_column] = np.where(df[target_column] == 'unacc', 0, 1)
y = df[target_column]

# Split the full dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=123, test_size=0.3)

# 1. Create a Gradient Boosting Classifier and print its parameters
grad_classifier = GradientBoostingClassifier(n_estimators=15)

print(grad_classifier.get_params())

# 2. Fit the Gradient Boosted Trees Classifier to the training data and get the list of predictions
grad_classifier.fit(X_train, y_train)
y_pred = grad_classifier.predict(X_test)

# 3. Calculate the accuracy, precision, recall, and f1-score on the testing data
accuracy = accuracy_score(y_test, y_pred).round(4)
precision = precision_score(y_test, y_pred).round(4)
recall = recall_score(y_test, y_pred).round(4)
f1 = f1_score(y_test, y_pred).round(4)

print(f'Test set accuracy:\t{accuracy}')
print(f'Test set precision:\t{precision}')
print(f'Test set recall:\t{recall}')
print(f'Test set f1-score:\t{f1}')

# 4. Remove the comments from the following code block to print the confusion matrix

test_conf_matrix = pd.DataFrame(
   confusion_matrix(y_test, y_pred, labels=[1, 0]), 
   index=['actual yes', 'actual no'], 
   columns=['predicted yes', 'predicted no']
)

print(f'Confusion Matrix:\n{test_conf_matrix.to_string()}')
```

---